# Generate Task 2 embeddings — Google Colab

This runs in **4 steps**. Just do **Runtime → Run all** and paste your GitHub token when asked.
Each step prints its own progress so you can see exactly what is happening:

1. **Copy your repo into Colab** (needs your GitHub token)
2. **Install libraries**
3. **Create the embeddings** (downloads the 2 models, shows progress bars)
4. **Push the embeddings back to GitHub**

When step 4 finishes, go to your office laptop and run `git pull`.


## STEP 1 of 4 — Copy your repo into Colab


In [ ]:
import os
from getpass import getpass

USER = "sanjaykumarpushadapu"
REPO = "mtech"
ASS  = "2026-2027-Sem1/AIMLCZG521-ConversationalAI/ass-1"

print("STEP 1 of 4 — copying your GitHub repo into Colab")
print("A hidden input box appears below. Paste your GitHub token and press Enter.\n")
TOKEN = getpass("GitHub token: ").strip()

if not TOKEN:
    print("\n\u274c No token entered. Re-run this cell and paste your token.")
else:
    print("\n\u23f3 Cloning github.com/%s/%s ..." % (USER, REPO))
    rc = os.system("rm -rf %s && git clone --quiet https://%s@github.com/%s/%s.git 2>/tmp/clone_err" % (REPO, TOKEN, USER, REPO))
    err = open("/tmp/clone_err").read().replace(TOKEN, "***")  # never print the token
    if rc == 0 and os.path.isdir(REPO):
        os.chdir(os.path.join(REPO, ASS))
        print("\u2705 STEP 1 DONE — your repo is now in Colab.")
        print("   Current folder:", os.getcwd())
        print("   Files here    :", ", ".join(sorted(os.listdir("."))[:10]))
    else:
        print("\u274c STEP 1 FAILED. Git said:")
        print("   ", (err.strip()[:400] or "(no message)"))
        print("\nMost likely the token is wrong or missing the 'repo' permission.")
        print("Make one: GitHub -> Settings -> Developer settings ->")
        print("  Personal access tokens -> Tokens (classic) -> tick 'repo' -> Generate.")


## STEP 2 of 4 — Install libraries


In [ ]:
import sys, subprocess
print("STEP 2 of 4 — installing libraries (about 30-60 seconds) ...")
rc = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "sentence-transformers", "pandas", "numpy"]).returncode
print("\u2705 STEP 2 DONE — libraries ready." if rc == 0 else "\u274c install failed (rc=%d)" % rc)


## STEP 3 of 4 — Create the embeddings


In [ ]:
import os, json, time
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

print("STEP 3 of 4 — creating embeddings for both models\n")
RAW, OUT = "data/raw", "data/results"
os.makedirs(OUT, exist_ok=True)

print("\u23f3 Loading SciFact data from data/raw ...")
corpus  = pd.read_json(f"{RAW}/corpus.jsonl",  lines=True)
queries = pd.read_json(f"{RAW}/queries.jsonl", lines=True)
qrels   = pd.read_csv(f"{RAW}/qrels.tsv", sep="\t")
corpus["doc_id"]    = corpus["doc_id"].astype(str)
queries["query_id"] = queries["query_id"].astype(str)
qrels["query_id"]   = qrels["query_id"].astype(str)
queries = queries[queries["query_id"].isin(set(qrels["query_id"]))].reset_index(drop=True)
print(f"\u2705 Data loaded: {len(corpus)} documents, {len(queries)} queries\n")

MODELS = {
    "distilbert-base-uncased": "distilbert-base-uncased",
    "BAAI/bge-large-en-v1.5":  "BAAI__bge-large-en-v1.5",
}
timings = {}
for n, (name, safe) in enumerate(MODELS.items(), 1):
    print(f"--- Model {n} of 2: {name} ---")
    print("  \u23f3 downloading the model (first time only) ...")
    model = SentenceTransformer(name)
    print("  \u2705 model ready. Creating DOCUMENT vectors (progress bar below):")
    t0 = time.time()
    doc_emb = model.encode(corpus["text"].tolist(), show_progress_bar=True)
    timings[name] = round(time.time() - t0, 2)
    print("  \u2705 document vectors done. Creating QUERY vectors:")
    qry_emb = model.encode(queries["text"].tolist(), show_progress_bar=True)
    np.save(f"{OUT}/embeddings_{safe}.npy", doc_emb)
    np.save(f"{OUT}/query_embeddings_{safe}.npy", qry_emb)
    print(f"  \u2705 saved: documents={doc_emb.shape}, queries={qry_emb.shape}, time={timings[name]}s\n")

json.dump(timings, open(f"{OUT}/task2_timings.json", "w"), indent=2)
print("\u2705 STEP 3 DONE — all embeddings created and saved to data/results/.")


## STEP 4 of 4 — Push the embeddings back to GitHub


In [ ]:
import os
print("STEP 4 of 4 — sending the embeddings to your GitHub repo\n")
os.system('git config user.email "sanjaykumar.pushadapu@gmail.com"')
os.system('git config user.name "Sanjay Kumar Pushadapu"')
os.system("git add data/results/embeddings_*.npy data/results/query_embeddings_*.npy data/results/task2_timings.json")
print("Files ready to upload:")
os.system("git --no-pager diff --cached --name-only")
rc = os.system('git commit -q -m "Add Task 2 embeddings (generated in Google Colab)"')
if rc != 0:
    print("\u2139\ufe0f  Nothing new to commit (maybe already pushed) - that is fine.")
print("\u23f3 pushing to GitHub ...")
rc = os.system("git push 2>/tmp/push_err")
err = open("/tmp/push_err").read()
tok = globals().get("TOKEN", "")
if tok:
    err = err.replace(tok, "***")
if rc == 0:
    print("\n\u2705 STEP 4 DONE — the embeddings are now in your GitHub repo!")
    print("\n\U0001f449 On your OFFICE laptop, run:   git pull")
    print("   then run the notebook - Task 2 will say 'Loaded SAVED embeddings'.")
else:
    print("\n\u274c push failed. Git said:")
    print("   ", (err.strip()[:400] or "(no message)"))
